# MenoPhenotype — Notebook 5: Preprocessing

**Course stage covered:** Stage 6 (Data Preprocessing).
**Input:** `artifacts/baseline_engineered.csv`, `artifacts/split_ids.json`, `artifacts/engineering_meta.json`,
`artifacts/screener_engineered.csv`, `artifacts/screener_validation_ids.csv`.
**Output:** `artifacts/baseline_train_features.csv`, `artifacts/baseline_test_features.csv`,
`artifacts/baseline_preprocessor.joblib` (+ `meno_preproc.py`), `artifacts/screener_validation_features.csv`.

Every data-dependent step here (winsorising bounds, imputation medians, scaler mean/SD) is **fit on the
training split only**, using the IDs saved in Notebook 4; the test split is only ever transformed.


In [1]:
%%writefile meno_preproc.py
# Kept in its own module so the fitted pipeline can be pickled and reloaded later.
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

class Winsorizer(BaseEstimator, TransformerMixin):
    """Clip each column to [lower, upper] quantiles learned on the TRAINING data only."""
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper
    def fit(self, X, y=None):
        if hasattr(X, 'columns'):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        Xv = np.asarray(X, dtype=float)
        self.lo_ = np.nanquantile(Xv, self.lower, axis=0)
        self.hi_ = np.nanquantile(Xv, self.upper, axis=0)
        self.n_features_in_ = Xv.shape[1]
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lo_, self.hi_)
    def get_feature_names_out(self, input_features=None):
        if input_features is not None:
            return np.asarray(input_features, dtype=object)
        return np.asarray(getattr(self, 'feature_names_in_',
                                   [f'x{i}' for i in range(self.n_features_in_)]), dtype=object)


Writing meno_preproc.py


In [2]:
import json, warnings
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
import joblib
from meno_preproc import Winsorizer

warnings.filterwarnings('ignore')
ART = Path('artifacts')
qmeta = json.load(open(ART / 'quality_meta.json'))
emeta = json.load(open(ART / 'engineering_meta.json'))
split = json.load(open(ART / 'split_ids.json'))

feat = pd.read_csv(ART / 'baseline_engineered.csv').set_index('SWANID')
train_ids = [i for i in split['train_ids'] if i in feat.index]
test_ids  = [i for i in split['test_ids']  if i in feat.index]
Xtr_raw, Xte_raw = feat.loc[train_ids], feat.loc[test_ids]
print(f"train = {len(Xtr_raw):,}  test = {len(Xte_raw):,}")


train = 2,632  test = 658


## A. Pipeline design

| Block | Columns | Steps |
|---|---|---|
| `ord` | 18 symptoms, `cesd_mean`, `SPORTS0`, `AGE0` | median impute -> standardise |
| `hormone` | 6 hormones | `log1p` (per Notebook 3's skew finding) -> winsorise (train 1st/99th pct) -> median impute -> standardise |
| `anthro` | `BMI0`, `WHR` | winsorise -> median impute -> standardise |

**Why median imputation:** after Notebook 2's row-drop, remaining gaps are a handful of scattered answers per
column — a simple, stable, leakage-safe method is adequate; the median is robust to skew and, for ordinal
items, always returns a valid level. **Why standardise:** features live on very different scales (0-4 scores,
hormone ng/mL-pg/mL, BMI) and the planned clustering models (K-Means, GMM) are scale-sensitive.


In [3]:
HORM_COLS = qmeta['HORM_COLS']
ORD  = qmeta['SYMP_COLS'] + ['cesd_mean', 'SPORTS0', 'AGE0']
ANTH = ['BMI0', 'WHR']

def block(*steps): return Pipeline(list(steps))

pre = ColumnTransformer([
    ('ord',     block(('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())), ORD),
    ('hormone', block(('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
                      ('win', Winsorizer(0.01, 0.99)),
                      ('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())), HORM_COLS),
    ('anthro',  block(('win', Winsorizer(0.01, 0.99)),
                      ('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())), ANTH),
], verbose_feature_names_out=False).set_output(transform='pandas')

Xtr = pre.fit_transform(Xtr_raw)          # FIT on train only
Xte = pre.transform(Xte_raw)              # transform only
print("train:", Xtr.shape, "| test:", Xte.shape)
print("NaNs left  train/test:", int(Xtr.isna().sum().sum()), int(Xte.isna().sum().sum()))
print("Train means ~0, SDs ~1 -> max|mean|:", Xtr.mean().abs().max().round(3),
      "| SD range:", Xtr.std().min().round(2), "-", Xtr.std().max().round(2))


train: (2632, 29) | test: (658, 29)
NaNs left  train/test: 0 0
Train means ~0, SDs ~1 -> max|mean|: 0.0 | SD range: 1.0 - 1.0


In [4]:
# Effect of winsorising: how many training values were clipped, per hormone/anthro column
for name, cols in [('hormone', HORM_COLS), ('anthro', ANTH)]:
    w = pre.named_transformers_[name].named_steps['win']
    x = np.log1p(Xtr_raw[cols]) if name == 'hormone' else Xtr_raw[cols]
    clipped = ((x.values < w.lo_) | (x.values > w.hi_)).sum(axis=0)
    print(name, dict(zip(cols, clipped)))


hormone {'FSH0': np.int64(54), 'E2AVE0': np.int64(54), 'T0': np.int64(54), 'TSH0': np.int64(52), 'SHBG0': np.int64(54), 'DHAS0': np.int64(54)}
anthro {'BMI0': np.int64(52), 'WHR': np.int64(52)}


## B. Save Baseline artefacts + reload check

In [5]:
Xtr.to_csv(ART / 'baseline_train_features.csv')
Xte.to_csv(ART / 'baseline_test_features.csv')
joblib.dump(pre, ART / 'baseline_preprocessor.joblib')

# prove the saved pipeline reproduces the test matrix exactly after a reload
chk = joblib.load(ART / 'baseline_preprocessor.joblib').transform(Xte_raw)
print("Reload check (max abs diff):", float(np.abs(chk.values - Xte.values).max()))
print(sorted(p.name for p in ART.iterdir() if p.suffix in ('.csv', '.joblib') and 'baseline' in p.name))


Reload check (max abs diff): 0.0
['baseline_clean.csv', 'baseline_engineered.csv', 'baseline_posthoc_labels.csv', 'baseline_preprocessor.joblib', 'baseline_profiling.csv', 'baseline_test_features.csv', 'baseline_train_features.csv']


## C. Screener validation cohort: matching preprocessing

The Screener's 11 items are already binary (0/1), so there is no scaling/winsorising to fit — only imputation
for the rare missing cell, using the mode (most frequent category), which is the appropriate leakage-safe
choice for a binary variable.


In [6]:
from sklearn.impute import SimpleImputer as SI
scr_feat = pd.read_csv(ART / 'screener_engineered.csv').set_index('SWANID')
val_ids = pd.read_csv(ART / 'screener_validation_ids.csv')['SWANID']
SCR_SYMP = qmeta['SCR_SYMP']

sv_raw = scr_feat.loc[val_ids, SCR_SYMP]
sv = pd.DataFrame(SI(strategy='most_frequent').fit_transform(sv_raw), index=sv_raw.index, columns=SCR_SYMP).astype(int)
sv.to_csv(ART / 'screener_validation_features.csv')
print(sv.shape, "-> saved screener_validation_features.csv")


(8713, 11) -> saved screener_validation_features.csv


### Summary

| Item | Detail |
|---|---|
| Baseline pipeline | log1p (hormones) -> winsorise (1st/99th pct, train-fit) -> median impute -> standardise |
| Fit strategy | `fit_transform` on train, `transform`-only on test; verified by reloading the saved `.joblib` |
| Screener | binary items, mode-imputed only (no scaling needed) |
| Files saved | `baseline_train_features.csv`, `baseline_test_features.csv`, `baseline_preprocessor.joblib`, `meno_preproc.py`, `screener_validation_features.csv` |
| Next notebook | Notebook 6 — Feature Selection |
